## Step 5: Feature Extraction — PCA for all three datasets

Both CIFAR-10 and MNIST use PCA + MLP. CIFAR-10 uses PCA-512 (richer features for the more complex dataset); MNIST uses PCA-256 (sufficient for grayscale digits). This unified architecture isolates the defense contribution from architectural confounds and is consistent with how foundational FL defense papers evaluate their methods.

In [ ]:
from sklearn.decomposition import PCA

# ─── v2: CIFAR-10/100 → flatten + PCA + standardise (MLP-ready) ───
def flatten_and_normalise(X):
    """Flatten (N,3,32,32) -> (N,3072), normalise to [-1,1] range."""
    X = X.reshape(len(X), -1).astype(np.float32)
    X = (X - 0.5) / 0.5
    return X

def extract_features(X_tr, X_te, n_comp, name=''):
    print(f'  [{name}] PCA({n_comp}) on {len(X_tr):,} samples…', end=' ')
    pca = PCA(n_components=n_comp, random_state=SEED, svd_solver='randomized')
    Xf_tr = pca.fit_transform(X_tr)
    Xf_te = pca.transform(X_te)
    var   = pca.explained_variance_ratio_.sum()
    print(f'variance explained: {var*100:.1f}%')
    return Xf_tr, Xf_te, pca

def standardise(Xtr, Xte):
    mu = Xtr.mean(0); sd = Xtr.std(0) + 1e-8
    return (Xtr - mu) / sd, (Xte - mu) / sd

# CIFAR-10 — flatten then PCA-512
Xc_flat      = flatten_and_normalise(X_cifar)
Xc_test_flat = flatten_and_normalise(X_cifar_test)
Xf_cifar, Xf_cifar_test, pca_cifar = extract_features(
    Xc_flat, Xc_test_flat, CIFAR_PCA_DIM, name='CIFAR-10')
Xf_cifar, Xf_cifar_test = standardise(Xf_cifar, Xf_cifar_test)

# CIFAR-100 (v7 NEW) — same pipeline as CIFAR-10, 100 fine-grained classes
Xc100_flat      = flatten_and_normalise(X_cifar100)
Xc100_test_flat = flatten_and_normalise(X_cifar100_test)
Xf_cifar100, Xf_cifar100_test, pca_cifar100 = extract_features(
    Xc100_flat, Xc100_test_flat, CIFAR100_PCA_DIM, name='CIFAR-100')
Xf_cifar100, Xf_cifar100_test = standardise(Xf_cifar100, Xf_cifar100_test)

# MNIST — l2-normalise rows then PCA-256
Xm_norm  = X_mnist      / (np.linalg.norm(X_mnist,      axis=1, keepdims=True) + 1e-8)
Xmt_norm = X_mnist_test / (np.linalg.norm(X_mnist_test, axis=1, keepdims=True) + 1e-8)
Xf_mnist, Xf_mnist_test, pca_mnist = extract_features(
    Xm_norm, Xmt_norm, MNIST_PCA_DIM, name='MNIST   ')
Xf_mnist, Xf_mnist_test = standardise(Xf_mnist, Xf_mnist_test)

print()
print(f'✅ CIFAR-10  features: PCA-{CIFAR_PCA_DIM} → {Xf_cifar.shape}  [MLP input]')
print(f'✅ CIFAR-100 features: PCA-{CIFAR100_PCA_DIM} → {Xf_cifar100.shape}  [MLP input]')
print(f'✅ MNIST     features: PCA-{MNIST_PCA_DIM} → {Xf_mnist.shape}  [MLP input]')
print(f'   All three datasets share the same MLP architecture for unified evaluation.')


  [CIFAR-10] PCA(512) on 30,000 samples… variance explained: 98.4%
  [CIFAR-100] PCA(512) on 30,000 samples… variance explained: 98.4%
  [MNIST   ] PCA(256) on 30,000 samples… variance explained: 97.9%

✅ CIFAR-10  features: PCA-512 → (30000, 512)  [MLP input]
✅ CIFAR-100 features: PCA-512 → (30000, 512)  [MLP input]
✅ MNIST     features: PCA-256 → (30000, 256)  [MLP input]
   All three datasets share the same MLP architecture for unified evaluation.
